# Personal Health Optimization Agent

This notebook implements a comprehensive personal health optimization system using HealthKit-like data processing and ML-powered health insights. The agent analyzes various health metrics, provides personalized recommendations, and tracks progress over time using advanced health analytics libraries.

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn matplotlib seaborn scipy statsmodels
!pip install lifelines  # For survival analysis
!pip install hmmlearn  # For Hidden Markov Models
!pip install dtaidistance  # For time series analysis
!pip install prophet  # For time series forecasting

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from enum import Enum
import json
import warnings
warnings.filterwarnings('ignore')

# ML and analysis libraries
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy import stats
from scipy.signal import find_peaks
import statsmodels.api as sm

# Specialized health analysis
try:
    from lifelines import KaplanMeierFitter
    from hmmlearn import hmm
    from dtaidistance import dtw
    from prophet import Prophet
except ImportError:
    print("Some specialized libraries not available. Using basic analysis.")

In [ ]:
class HealthMetricType(Enum):
    """Types of health metrics"""
    HEART_RATE = "heart_rate"
    BLOOD_PRESSURE = "blood_pressure"
    SLEEP = "sleep"
    STEPS = "steps"
    NUTRITION = "nutrition"
    WEIGHT = "weight"
    BLOOD_GLUCOSE = "blood_glucose"
    STRESS = "stress"
    HYDRATION = "hydration"
    EXERCISE = "exercise"
    MOOD = "mood"
    TEMPERATURE = "temperature"
    OXYGEN_SATURATION = "oxygen_saturation"
    RESPIRATORY_RATE = "respiratory_rate"

@dataclass
class HealthMetric:
    """Individual health metric data point"""
    metric_type: HealthMetricType
    value: float
    timestamp: datetime
    unit: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    confidence: float = 1.0

@dataclass
class HealthInsight:
    """Health insight or recommendation"""
    category: str
    severity: str  # low, medium, high, critical
    title: str
    description: str
    recommendations: List[str]
    confidence: float
    evidence: Dict[str, Any] = field(default_factory=dict)
    action_items: List[str] = field(default_factory=list)

@dataclass
class HealthGoal:
    """Personal health goal"""
    goal_type: str
    target_value: float
    current_value: float
    deadline: datetime
    priority: str
    milestones: List[Dict[str, Any]] = field(default_factory=list)

In [ ]:
class HealthDataProcessor:
    """Process and analyze health data"""
    
    def __init__(self):
        self.scalers = {}
        self.models = {}
        self.thresholds = self._initialize_thresholds()
        
    def _initialize_thresholds(self) -> Dict[str, Dict[str, float]]:
        """Initialize health metric thresholds"""
        return {
            HealthMetricType.HEART_RATE.value: {
                "resting_min": 50, "resting_max": 100,
                "active_max": 180, "recovery_time": 300
            },
            HealthMetricType.BLOOD_PRESSURE.value: {
                "systolic_min": 90, "systolic_max": 140,
                "diastolic_min": 60, "diastolic_max": 90
            },
            HealthMetricType.SLEEP.value: {
                "min_hours": 6, "optimal_hours": 8,
                "deep_sleep_percent": 0.15, "rem_percent": 0.20
            },
            HealthMetricType.STEPS.value: {
                "daily_min": 5000, "daily_optimal": 10000,
                "hourly_active": 250
            },
            HealthMetricType.BLOOD_GLUCOSE.value: {
                "fasting_min": 70, "fasting_max": 100,
                "postprandial_max": 140
            },
            HealthMetricType.OXYGEN_SATURATION.value: {
                "min_normal": 95, "concerning": 92, "critical": 88
            }
        }
    
    def detect_anomalies(self, metrics: List[HealthMetric]) -> List[Dict[str, Any]]:
        """Detect anomalies in health metrics"""
        anomalies = []
        
        # Group metrics by type
        grouped = {}
        for metric in metrics:
            if metric.metric_type.value not in grouped:
                grouped[metric.metric_type.value] = []
            grouped[metric.metric_type.value].append(metric)
        
        # Analyze each metric type
        for metric_type, type_metrics in grouped.items():
            if len(type_metrics) < 10:
                continue
                
            values = np.array([m.value for m in type_metrics])
            timestamps = [m.timestamp for m in type_metrics]
            
            # Statistical anomaly detection
            z_scores = np.abs(stats.zscore(values))
            statistical_anomalies = np.where(z_scores > 3)[0]
            
            # Isolation Forest for complex patterns
            if len(values) > 20:
                iso_forest = IsolationForest(contamination=0.1, random_state=42)
                predictions = iso_forest.fit_predict(values.reshape(-1, 1))
                isolation_anomalies = np.where(predictions == -1)[0]
            else:
                isolation_anomalies = []
            
            # Threshold-based anomalies
            threshold_anomalies = self._check_thresholds(metric_type, values)
            
            # Combine all anomaly indices
            all_anomaly_indices = set(statistical_anomalies) | set(isolation_anomalies) | set(threshold_anomalies)
            
            for idx in all_anomaly_indices:
                anomalies.append({
                    "metric_type": metric_type,
                    "timestamp": timestamps[idx],
                    "value": values[idx],
                    "z_score": z_scores[idx] if idx < len(z_scores) else None,
                    "anomaly_types": [
                        "statistical" if idx in statistical_anomalies else None,
                        "pattern" if idx in isolation_anomalies else None,
                        "threshold" if idx in threshold_anomalies else None
                    ]
                })
        
        return anomalies
    
    def _check_thresholds(self, metric_type: str, values: np.ndarray) -> List[int]:
        """Check values against predefined thresholds"""
        anomalies = []
        if metric_type not in self.thresholds:
            return anomalies
            
        thresholds = self.thresholds[metric_type]
        
        for i, value in enumerate(values):
            if metric_type == HealthMetricType.HEART_RATE.value:
                if value < thresholds["resting_min"] or value > thresholds["active_max"]:
                    anomalies.append(i)
            elif metric_type == HealthMetricType.BLOOD_PRESSURE.value:
                # Assuming value is systolic for simplicity
                if value < thresholds["systolic_min"] or value > thresholds["systolic_max"]:
                    anomalies.append(i)
            elif metric_type == HealthMetricType.OXYGEN_SATURATION.value:
                if value < thresholds["critical"]:
                    anomalies.append(i)
        
        return anomalies
    
    def analyze_patterns(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Analyze patterns in health metrics"""
        patterns = {
            "circadian_rhythm": self._analyze_circadian_rhythm(metrics),
            "weekly_patterns": self._analyze_weekly_patterns(metrics),
            "correlations": self._analyze_correlations(metrics),
            "trends": self._analyze_trends(metrics)
        }
        return patterns
    
    def _analyze_circadian_rhythm(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Analyze 24-hour patterns"""
        hourly_data = {}
        
        for metric in metrics:
            hour = metric.timestamp.hour
            metric_type = metric.metric_type.value
            
            if metric_type not in hourly_data:
                hourly_data[metric_type] = {i: [] for i in range(24)}
            
            hourly_data[metric_type][hour].append(metric.value)
        
        # Calculate hourly averages and patterns
        patterns = {}
        for metric_type, hours in hourly_data.items():
            hourly_avg = {h: np.mean(vals) if vals else None for h, vals in hours.items()}
            
            # Find peaks and troughs
            values = [v for v in hourly_avg.values() if v is not None]
            if len(values) > 12:
                peaks, _ = find_peaks(values)
                troughs, _ = find_peaks([-v for v in values])
                
                patterns[metric_type] = {
                    "hourly_average": hourly_avg,
                    "peak_hours": peaks.tolist(),
                    "trough_hours": troughs.tolist(),
                    "amplitude": max(values) - min(values) if values else 0
                }
        
        return patterns
    
    def _analyze_weekly_patterns(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Analyze weekly patterns"""
        weekly_data = {}
        
        for metric in metrics:
            day = metric.timestamp.weekday()
            metric_type = metric.metric_type.value
            
            if metric_type not in weekly_data:
                weekly_data[metric_type] = {i: [] for i in range(7)}
            
            weekly_data[metric_type][day].append(metric.value)
        
        patterns = {}
        for metric_type, days in weekly_data.items():
            daily_avg = {d: np.mean(vals) if vals else None for d, vals in days.items()}
            
            patterns[metric_type] = {
                "daily_average": daily_avg,
                "weekend_vs_weekday": self._compare_weekend_weekday(daily_avg)
            }
        
        return patterns
    
    def _compare_weekend_weekday(self, daily_avg: Dict[int, float]) -> Dict[str, float]:
        """Compare weekend vs weekday metrics"""
        weekday_vals = [daily_avg[i] for i in range(5) if daily_avg.get(i) is not None]
        weekend_vals = [daily_avg[i] for i in [5, 6] if daily_avg.get(i) is not None]
        
        if weekday_vals and weekend_vals:
            return {
                "weekday_avg": np.mean(weekday_vals),
                "weekend_avg": np.mean(weekend_vals),
                "difference": np.mean(weekend_vals) - np.mean(weekday_vals)
            }
        return {}
    
    def _analyze_correlations(self, metrics: List[HealthMetric]) -> Dict[str, float]:
        """Analyze correlations between different metrics"""
        # Create DataFrame for correlation analysis
        data = {}
        for metric in metrics:
            key = f"{metric.metric_type.value}_{metric.timestamp.date()}"
            if key not in data:
                data[key] = metric.value
        
        if len(data) < 10:
            return {}
        
        # Convert to DataFrame and calculate correlations
        df = pd.DataFrame(list(data.items()), columns=['metric_date', 'value'])
        df['metric_type'] = df['metric_date'].apply(lambda x: x.split('_')[0])
        df['date'] = pd.to_datetime(df['metric_date'].apply(lambda x: x.split('_')[1]))
        
        # Pivot to get metrics as columns
        pivot_df = df.pivot_table(index='date', columns='metric_type', values='value')
        
        if pivot_df.shape[1] > 1:
            correlations = pivot_df.corr()
            
            # Extract significant correlations
            significant_corr = {}
            for i in range(len(correlations.columns)):
                for j in range(i+1, len(correlations.columns)):
                    corr_value = correlations.iloc[i, j]
                    if abs(corr_value) > 0.5 and not np.isnan(corr_value):
                        key = f"{correlations.columns[i]}_vs_{correlations.columns[j]}"
                        significant_corr[key] = corr_value
            
            return significant_corr
        
        return {}
    
    def _analyze_trends(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Analyze trends over time"""
        trends = {}
        
        # Group by metric type
        grouped = {}
        for metric in metrics:
            if metric.metric_type.value not in grouped:
                grouped[metric.metric_type.value] = []
            grouped[metric.metric_type.value].append(metric)
        
        for metric_type, type_metrics in grouped.items():
            if len(type_metrics) < 7:
                continue
            
            # Sort by timestamp
            type_metrics.sort(key=lambda x: x.timestamp)
            
            values = [m.value for m in type_metrics]
            timestamps = [m.timestamp for m in type_metrics]
            
            # Calculate moving averages
            ma_7 = pd.Series(values).rolling(window=7).mean().tolist()
            ma_30 = pd.Series(values).rolling(window=30).mean().tolist()
            
            # Linear regression for trend
            x = np.arange(len(values))
            slope, intercept = np.polyfit(x, values, 1)
            
            # Determine trend direction
            if abs(slope) < 0.01:
                trend_direction = "stable"
            elif slope > 0:
                trend_direction = "increasing"
            else:
                trend_direction = "decreasing"
            
            trends[metric_type] = {
                "direction": trend_direction,
                "slope": float(slope),
                "ma_7_latest": ma_7[-1] if ma_7 and not np.isnan(ma_7[-1]) else None,
                "ma_30_latest": ma_30[-1] if ma_30 and not np.isnan(ma_30[-1]) else None,
                "volatility": float(np.std(values)),
                "recent_change": float(values[-1] - values[0]) if values else 0
            }
        
        return trends

In [ ]:
class HealthInsightGenerator:
    """Generate actionable health insights"""
    
    def __init__(self):
        self.processor = HealthDataProcessor()
        
    def generate_insights(self, metrics: List[HealthMetric], 
                         user_profile: Dict[str, Any]) -> List[HealthInsight]:
        """Generate comprehensive health insights"""
        insights = []
        
        # Analyze anomalies
        anomalies = self.processor.detect_anomalies(metrics)
        if anomalies:
            insights.extend(self._generate_anomaly_insights(anomalies))
        
        # Analyze patterns
        patterns = self.processor.analyze_patterns(metrics)
        insights.extend(self._generate_pattern_insights(patterns))
        
        # Generate metric-specific insights
        insights.extend(self._generate_metric_insights(metrics, user_profile))
        
        # Generate holistic health insights
        insights.extend(self._generate_holistic_insights(metrics, patterns))
        
        # Prioritize insights
        return sorted(insights, key=lambda x: self._priority_score(x), reverse=True)
    
    def _generate_anomaly_insights(self, anomalies: List[Dict[str, Any]]) -> List[HealthInsight]:
        """Generate insights from anomalies"""
        insights = []
        
        # Group anomalies by type
        grouped = {}
        for anomaly in anomalies:
            metric_type = anomaly['metric_type']
            if metric_type not in grouped:
                grouped[metric_type] = []
            grouped[metric_type].append(anomaly)
        
        for metric_type, type_anomalies in grouped.items():
            if metric_type == HealthMetricType.HEART_RATE.value:
                insights.append(self._heart_rate_anomaly_insight(type_anomalies))
            elif metric_type == HealthMetricType.SLEEP.value:
                insights.append(self._sleep_anomaly_insight(type_anomalies))
            elif metric_type == HealthMetricType.BLOOD_PRESSURE.value:
                insights.append(self._blood_pressure_anomaly_insight(type_anomalies))
        
        return [i for i in insights if i is not None]
    
    def _heart_rate_anomaly_insight(self, anomalies: List[Dict[str, Any]]) -> Optional[HealthInsight]:
        """Generate heart rate anomaly insight"""
        if not anomalies:
            return None
            
        high_values = [a for a in anomalies if a['value'] > 100]
        low_values = [a for a in anomalies if a['value'] < 50]
        
        if high_values:
            return HealthInsight(
                category="Cardiovascular",
                severity="high" if any(a['value'] > 150 for a in high_values) else "medium",
                title="Elevated Heart Rate Detected",
                description=f"Your heart rate has been elevated {len(high_values)} times in the monitoring period.",
                recommendations=[
                    "Consider stress reduction techniques",
                    "Ensure adequate hydration",
                    "Review caffeine intake",
                    "Consult healthcare provider if persistent"
                ],
                confidence=0.85,
                evidence={"anomaly_count": len(high_values), "max_value": max(a['value'] for a in high_values)},
                action_items=[
                    "Monitor heart rate more frequently",
                    "Track activities during elevated periods"
                ]
            )
        elif low_values:
            return HealthInsight(
                category="Cardiovascular",
                severity="medium",
                title="Low Heart Rate Detected",
                description=f"Your heart rate has been unusually low {len(low_values)} times.",
                recommendations=[
                    "Ensure you're getting adequate nutrition",
                    "Check for signs of fatigue or dizziness",
                    "Consider medical evaluation"
                ],
                confidence=0.80,
                evidence={"anomaly_count": len(low_values), "min_value": min(a['value'] for a in low_values)}
            )
        
        return None
    
    def _sleep_anomaly_insight(self, anomalies: List[Dict[str, Any]]) -> Optional[HealthInsight]:
        """Generate sleep anomaly insight"""
        if not anomalies:
            return None
            
        low_sleep = [a for a in anomalies if a['value'] < 6]
        
        if low_sleep:
            return HealthInsight(
                category="Sleep",
                severity="high" if any(a['value'] < 4 for a in low_sleep) else "medium",
                title="Insufficient Sleep Detected",
                description=f"You've had insufficient sleep {len(low_sleep)} times recently.",
                recommendations=[
                    "Establish consistent sleep schedule",
                    "Create bedtime routine",
                    "Limit screen time before bed",
                    "Optimize sleep environment"
                ],
                confidence=0.90,
                evidence={"low_sleep_nights": len(low_sleep)},
                action_items=[
                    "Set sleep reminder for 10 PM",
                    "Track sleep quality factors"
                ]
            )
        
        return None
    
    def _blood_pressure_anomaly_insight(self, anomalies: List[Dict[str, Any]]) -> Optional[HealthInsight]:
        """Generate blood pressure anomaly insight"""
        if not anomalies:
            return None
            
        high_bp = [a for a in anomalies if a['value'] > 140]
        
        if high_bp:
            return HealthInsight(
                category="Cardiovascular",
                severity="high",
                title="Elevated Blood Pressure Detected",
                description=f"Your blood pressure has been elevated {len(high_bp)} times.",
                recommendations=[
                    "Reduce sodium intake",
                    "Increase physical activity",
                    "Practice stress management",
                    "Consult healthcare provider"
                ],
                confidence=0.85,
                evidence={"high_readings": len(high_bp)},
                action_items=[
                    "Monitor blood pressure daily",
                    "Keep blood pressure log"
                ]
            )
        
        return None
    
    def _generate_pattern_insights(self, patterns: Dict[str, Any]) -> List[HealthInsight]:
        """Generate insights from patterns"""
        insights = []
        
        # Circadian rhythm insights
        if 'circadian_rhythm' in patterns:
            for metric_type, rhythm in patterns['circadian_rhythm'].items():
                if rhythm and 'amplitude' in rhythm:
                    if rhythm['amplitude'] > 20:  # High variability
                        insights.append(HealthInsight(
                            category="Lifestyle",
                            severity="low",
                            title=f"High Daily Variability in {metric_type.replace('_', ' ').title()}",
                            description="Your metrics show significant variation throughout the day.",
                            recommendations=[
                                "Maintain consistent daily routines",
                                "Regular meal times may help"
                            ],
                            confidence=0.75,
                            evidence={"amplitude": rhythm['amplitude']}
                        ))
        
        # Weekly pattern insights
        if 'weekly_patterns' in patterns:
            for metric_type, weekly in patterns['weekly_patterns'].items():
                if 'weekend_vs_weekday' in weekly and weekly['weekend_vs_weekday']:
                    diff = weekly['weekend_vs_weekday'].get('difference', 0)
                    if abs(diff) > 10:
                        insights.append(HealthInsight(
                            category="Lifestyle",
                            severity="low",
                            title=f"Weekend vs Weekday Difference in {metric_type.replace('_', ' ').title()}",
                            description=f"Your {metric_type} differs by {abs(diff):.1f} between weekdays and weekends.",
                            recommendations=[
                                "Try to maintain consistent habits throughout the week",
                                "Plan weekend activities that support your health goals"
                            ],
                            confidence=0.70,
                            evidence={"difference": diff}
                        ))
        
        # Correlation insights
        if 'correlations' in patterns and patterns['correlations']:
            for correlation, value in patterns['correlations'].items():
                metrics = correlation.split('_vs_')
                if len(metrics) == 2:
                    insights.append(HealthInsight(
                        category="Patterns",
                        severity="low",
                        title=f"Strong Correlation: {metrics[0].title()} and {metrics[1].title()}",
                        description=f"These metrics show a {'positive' if value > 0 else 'negative'} correlation of {abs(value):.2f}.",
                        recommendations=[
                            f"Consider how changes in {metrics[0]} might affect {metrics[1]}",
                            "Use this relationship to optimize both metrics"
                        ],
                        confidence=0.65,
                        evidence={"correlation": value}
                    ))
        
        return insights
    
    def _generate_metric_insights(self, metrics: List[HealthMetric], 
                                 user_profile: Dict[str, Any]) -> List[HealthInsight]:
        """Generate insights specific to each metric type"""
        insights = []
        
        # Group metrics by type
        grouped = {}
        for metric in metrics:
            if metric.metric_type.value not in grouped:
                grouped[metric.metric_type.value] = []
            grouped[metric.metric_type.value].append(metric)
        
        # Steps insights
        if HealthMetricType.STEPS.value in grouped:
            steps_data = grouped[HealthMetricType.STEPS.value]
            avg_steps = np.mean([m.value for m in steps_data])
            
            if avg_steps < 5000:
                insights.append(HealthInsight(
                    category="Activity",
                    severity="medium",
                    title="Low Daily Step Count",
                    description=f"Your average daily steps ({avg_steps:.0f}) are below recommended levels.",
                    recommendations=[
                        "Set hourly movement reminders",
                        "Take walking meetings",
                        "Use stairs instead of elevators",
                        "Park farther from entrances"
                    ],
                    confidence=0.85,
                    evidence={"avg_steps": avg_steps},
                    action_items=[
                        "Set daily step goal of 7,000",
                        "Track progress weekly"
                    ]
                ))
        
        # Hydration insights
        if HealthMetricType.HYDRATION.value in grouped:
            hydration_data = grouped[HealthMetricType.HYDRATION.value]
            avg_hydration = np.mean([m.value for m in hydration_data])
            
            if avg_hydration < 2000:  # Less than 2 liters
                insights.append(HealthInsight(
                    category="Nutrition",
                    severity="medium",
                    title="Insufficient Hydration",
                    description="Your daily water intake is below recommended levels.",
                    recommendations=[
                        "Keep water bottle visible",
                        "Set hydration reminders",
                        "Drink water before each meal",
                        "Monitor urine color"
                    ],
                    confidence=0.80,
                    evidence={"avg_ml": avg_hydration}
                ))
        
        return insights
    
    def _generate_holistic_insights(self, metrics: List[HealthMetric], 
                                   patterns: Dict[str, Any]) -> List[HealthInsight]:
        """Generate insights from overall health picture"""
        insights = []
        
        # Calculate overall health score
        health_score = self._calculate_health_score(metrics)
        
        if health_score < 70:
            insights.append(HealthInsight(
                category="Overall Health",
                severity="medium",
                title="Room for Health Improvement",
                description=f"Your overall health score is {health_score:.0f}/100.",
                recommendations=[
                    "Focus on consistent sleep schedule",
                    "Increase daily physical activity",
                    "Improve nutrition quality",
                    "Manage stress levels"
                ],
                confidence=0.75,
                evidence={"health_score": health_score},
                action_items=[
                    "Set 3 health goals for next month",
                    "Track progress weekly"
                ]
            ))
        elif health_score > 85:
            insights.append(HealthInsight(
                category="Overall Health",
                severity="low",
                title="Excellent Health Status",
                description=f"Your overall health score is {health_score:.0f}/100. Keep up the great work!",
                recommendations=[
                    "Maintain current healthy habits",
                    "Consider new fitness challenges",
                    "Share success with others"
                ],
                confidence=0.80,
                evidence={"health_score": health_score}
            ))
        
        return insights
    
    def _calculate_health_score(self, metrics: List[HealthMetric]) -> float:
        """Calculate overall health score"""
        scores = []
        
        # Group metrics by type
        grouped = {}
        for metric in metrics:
            if metric.metric_type.value not in grouped:
                grouped[metric.metric_type.value] = []
            grouped[metric.metric_type.value].append(metric)
        
        # Calculate scores for each metric type
        if HealthMetricType.STEPS.value in grouped:
            avg_steps = np.mean([m.value for m in grouped[HealthMetricType.STEPS.value]])
            step_score = min(100, (avg_steps / 10000) * 100)
            scores.append(step_score)
        
        if HealthMetricType.SLEEP.value in grouped:
            avg_sleep = np.mean([m.value for m in grouped[HealthMetricType.SLEEP.value]])
            sleep_score = min(100, (avg_sleep / 8) * 100)
            scores.append(sleep_score)
        
        if HealthMetricType.HEART_RATE.value in grouped:
            hr_values = [m.value for m in grouped[HealthMetricType.HEART_RATE.value]]
            # Assuming resting heart rate, lower is better within range
            avg_hr = np.mean(hr_values)
            if 50 <= avg_hr <= 70:
                hr_score = 100
            elif avg_hr < 50:
                hr_score = 80
            else:
                hr_score = max(0, 100 - (avg_hr - 70) * 2)
            scores.append(hr_score)
        
        return np.mean(scores) if scores else 50
    
    def _priority_score(self, insight: HealthInsight) -> float:
        """Calculate priority score for an insight"""
        severity_scores = {
            "critical": 4.0,
            "high": 3.0,
            "medium": 2.0,
            "low": 1.0
        }
        
        base_score = severity_scores.get(insight.severity, 1.0)
        confidence_factor = insight.confidence
        action_factor = 1.2 if insight.action_items else 1.0
        
        return base_score * confidence_factor * action_factor

In [ ]:
class PersonalHealthOptimizationAgent:
    """Main agent for personal health optimization"""
    
    def __init__(self):
        self.insight_generator = HealthInsightGenerator()
        self.processor = HealthDataProcessor()
        self.goals = []
        self.user_profile = {}
        
    def analyze_health_data(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Analyze health data and provide comprehensive report"""
        # Generate insights
        insights = self.insight_generator.generate_insights(metrics, self.user_profile)
        
        # Detect anomalies
        anomalies = self.processor.detect_anomalies(metrics)
        
        # Analyze patterns
        patterns = self.processor.analyze_patterns(metrics)
        
        # Generate recommendations
        recommendations = self._generate_recommendations(insights, patterns)
        
        # Create health report
        report = {
            "summary": self._generate_summary(metrics, insights),
            "insights": insights,
            "anomalies": anomalies,
            "patterns": patterns,
            "recommendations": recommendations,
            "goals": self._suggest_goals(insights, patterns),
            "risk_assessment": self._assess_health_risks(metrics, anomalies),
            "progress": self._track_progress(metrics)
        }
        
        return report
    
    def _generate_summary(self, metrics: List[HealthMetric], 
                         insights: List[HealthInsight]) -> Dict[str, Any]:
        """Generate executive summary"""
        health_score = self.insight_generator._calculate_health_score(metrics)
        
        critical_insights = [i for i in insights if i.severity == "critical"]
        high_insights = [i for i in insights if i.severity == "high"]
        
        return {
            "overall_health_score": health_score,
            "status": self._get_health_status(health_score),
            "critical_issues": len(critical_insights),
            "high_priority_issues": len(high_insights),
            "total_insights": len(insights),
            "monitored_metrics": len(set(m.metric_type for m in metrics)),
            "data_points": len(metrics),
            "key_findings": [i.title for i in insights[:3]]  # Top 3 insights
        }
    
    def _get_health_status(self, score: float) -> str:
        """Get health status from score"""
        if score >= 90:
            return "Excellent"
        elif score >= 80:
            return "Good"
        elif score >= 70:
            return "Fair"
        elif score >= 60:
            return "Needs Improvement"
        else:
            return "Poor"
    
    def _generate_recommendations(self, insights: List[HealthInsight], 
                                patterns: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Generate prioritized recommendations"""
        recommendations = []
        
        # Collect all recommendations from insights
        for insight in insights:
            for rec in insight.recommendations:
                recommendations.append({
                    "recommendation": rec,
                    "category": insight.category,
                    "priority": insight.severity,
                    "related_insight": insight.title
                })
        
        # Add pattern-based recommendations
        if 'trends' in patterns:
            for metric_type, trend in patterns['trends'].items():
                if trend['direction'] == "decreasing" and metric_type == HealthMetricType.STEPS.value:
                    recommendations.append({
                        "recommendation": "Gradually increase daily step count by 500 steps per week",
                        "category": "Activity",
                        "priority": "medium",
                        "related_insight": "Declining activity trend"
                    })
        
        # Remove duplicates and prioritize
        unique_recs = {}
        for rec in recommendations:
            if rec['recommendation'] not in unique_recs:
                unique_recs[rec['recommendation']] = rec
        
        return sorted(unique_recs.values(), 
                     key=lambda x: {'critical': 4, 'high': 3, 'medium': 2, 'low': 1}.get(x['priority'], 0),
                     reverse=True)
    
    def _suggest_goals(self, insights: List[HealthInsight], 
                      patterns: Dict[str, Any]) -> List[HealthGoal]:
        """Suggest personalized health goals"""
        goals = []
        
        # Goal suggestions based on insights
        for insight in insights[:5]:  # Top 5 insights
            if insight.category == "Activity" and "step" in insight.title.lower():
                goals.append(HealthGoal(
                    goal_type="Daily Steps",
                    target_value=10000,
                    current_value=5000,  # Would be calculated from actual data
                    deadline=datetime.now() + timedelta(days=30),
                    priority="high",
                    milestones=[
                        {"week": 1, "target": 6000},
                        {"week": 2, "target": 7500},
                        {"week": 3, "target": 9000},
                        {"week": 4, "target": 10000}
                    ]
                ))
            elif insight.category == "Sleep":
                goals.append(HealthGoal(
                    goal_type="Sleep Duration",
                    target_value=8,
                    current_value=6,
                    deadline=datetime.now() + timedelta(days=21),
                    priority="high",
                    milestones=[
                        {"week": 1, "target": 6.5},
                        {"week": 2, "target": 7.5},
                        {"week": 3, "target": 8}
                    ]
                ))
        
        return goals
    
    def _assess_health_risks(self, metrics: List[HealthMetric], 
                           anomalies: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Assess health risks based on data"""
        risks = {
            "cardiovascular": self._assess_cardiovascular_risk(metrics, anomalies),
            "metabolic": self._assess_metabolic_risk(metrics),
            "sleep_disorder": self._assess_sleep_risk(metrics),
            "stress": self._assess_stress_risk(metrics),
            "overall_risk_level": "low"  # Default
        }
        
        # Calculate overall risk
        risk_scores = [r['score'] for r in risks.values() if isinstance(r, dict) and 'score' in r]
        if risk_scores:
            avg_risk = np.mean(risk_scores)
            if avg_risk > 0.7:
                risks['overall_risk_level'] = "high"
            elif avg_risk > 0.4:
                risks['overall_risk_level'] = "medium"
        
        return risks
    
    def _assess_cardiovascular_risk(self, metrics: List[HealthMetric], 
                                  anomalies: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Assess cardiovascular risk"""
        risk_factors = 0
        evidence = []
        
        # Check heart rate anomalies
        hr_anomalies = [a for a in anomalies if a['metric_type'] == HealthMetricType.HEART_RATE.value]
        if len(hr_anomalies) > 5:
            risk_factors += 1
            evidence.append("Frequent heart rate anomalies")
        
        # Check blood pressure
        bp_metrics = [m for m in metrics if m.metric_type == HealthMetricType.BLOOD_PRESSURE]
        if bp_metrics:
            avg_bp = np.mean([m.value for m in bp_metrics])
            if avg_bp > 130:
                risk_factors += 2
                evidence.append("Elevated blood pressure")
        
        # Check activity levels
        steps_metrics = [m for m in metrics if m.metric_type == HealthMetricType.STEPS]
        if steps_metrics:
            avg_steps = np.mean([m.value for m in steps_metrics])
            if avg_steps < 5000:
                risk_factors += 1
                evidence.append("Low physical activity")
        
        risk_score = min(1.0, risk_factors / 4)
        
        return {
            "score": risk_score,
            "level": "high" if risk_score > 0.6 else "medium" if risk_score > 0.3 else "low",
            "factors": evidence,
            "recommendations": [
                "Regular cardiovascular screening",
                "Increase aerobic exercise",
                "Monitor blood pressure daily"
            ] if risk_score > 0.3 else []
        }
    
    def _assess_metabolic_risk(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Assess metabolic risk"""
        risk_factors = 0
        evidence = []
        
        # Check weight trends
        weight_metrics = [m for m in metrics if m.metric_type == HealthMetricType.WEIGHT]
        if weight_metrics and len(weight_metrics) > 7:
            weights = [m.value for m in sorted(weight_metrics, key=lambda x: x.timestamp)]
            weight_change = weights[-1] - weights[0]
            if weight_change > 2:  # Gained more than 2 kg
                risk_factors += 1
                evidence.append("Weight gain trend")
        
        # Check glucose if available
        glucose_metrics = [m for m in metrics if m.metric_type == HealthMetricType.BLOOD_GLUCOSE]
        if glucose_metrics:
            avg_glucose = np.mean([m.value for m in glucose_metrics])
            if avg_glucose > 100:
                risk_factors += 2
                evidence.append("Elevated blood glucose")
        
        risk_score = min(1.0, risk_factors / 3)
        
        return {
            "score": risk_score,
            "level": "high" if risk_score > 0.6 else "medium" if risk_score > 0.3 else "low",
            "factors": evidence
        }
    
    def _assess_sleep_risk(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Assess sleep disorder risk"""
        sleep_metrics = [m for m in metrics if m.metric_type == HealthMetricType.SLEEP]
        
        if not sleep_metrics:
            return {"score": 0, "level": "unknown"}
        
        risk_factors = 0
        evidence = []
        
        sleep_hours = [m.value for m in sleep_metrics]
        avg_sleep = np.mean(sleep_hours)
        sleep_variance = np.var(sleep_hours)
        
        if avg_sleep < 6:
            risk_factors += 2
            evidence.append("Chronic sleep deprivation")
        
        if sleep_variance > 2:
            risk_factors += 1
            evidence.append("Irregular sleep pattern")
        
        risk_score = min(1.0, risk_factors / 3)
        
        return {
            "score": risk_score,
            "level": "high" if risk_score > 0.6 else "medium" if risk_score > 0.3 else "low",
            "factors": evidence
        }
    
    def _assess_stress_risk(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Assess stress risk based on various indicators"""
        risk_factors = 0
        evidence = []
        
        # Heart rate variability (simplified)
        hr_metrics = [m for m in metrics if m.metric_type == HealthMetricType.HEART_RATE]
        if hr_metrics and len(hr_metrics) > 10:
            hr_values = [m.value for m in hr_metrics]
            hr_variance = np.var(hr_values)
            if hr_variance < 5:  # Low HRV
                risk_factors += 1
                evidence.append("Low heart rate variability")
        
        # Sleep quality
        sleep_metrics = [m for m in metrics if m.metric_type == HealthMetricType.SLEEP]
        if sleep_metrics:
            avg_sleep = np.mean([m.value for m in sleep_metrics])
            if avg_sleep < 7:
                risk_factors += 1
                evidence.append("Poor sleep quality")
        
        risk_score = min(1.0, risk_factors / 2)
        
        return {
            "score": risk_score,
            "level": "high" if risk_score > 0.6 else "medium" if risk_score > 0.3 else "low",
            "factors": evidence
        }
    
    def _track_progress(self, metrics: List[HealthMetric]) -> Dict[str, Any]:
        """Track progress towards goals"""
        progress = {}
        
        for goal in self.goals:
            relevant_metrics = [m for m in metrics 
                              if m.metric_type.value.lower() in goal.goal_type.lower()]
            
            if relevant_metrics:
                current_value = np.mean([m.value for m in relevant_metrics[-7:]])  # Last week
                
                progress[goal.goal_type] = {
                    "current": current_value,
                    "target": goal.target_value,
                    "progress_percent": ((current_value - goal.current_value) / 
                                       (goal.target_value - goal.current_value) * 100),
                    "on_track": current_value >= goal.milestones[0]['target'] if goal.milestones else True
                }
        
        return progress
    
    def generate_daily_briefing(self, metrics: List[HealthMetric]) -> str:
        """Generate a daily health briefing"""
        # Get yesterday's metrics
        yesterday = datetime.now() - timedelta(days=1)
        yesterday_metrics = [m for m in metrics 
                           if m.timestamp.date() == yesterday.date()]
        
        if not yesterday_metrics:
            return "No data available for yesterday."
        
        briefing = ["📊 Daily Health Briefing\n"]
        
        # Sleep summary
        sleep_data = [m for m in yesterday_metrics if m.metric_type == HealthMetricType.SLEEP]
        if sleep_data:
            sleep_hours = sleep_data[0].value
            briefing.append(f"😴 Sleep: {sleep_hours:.1f} hours")
        
        # Activity summary
        steps_data = [m for m in yesterday_metrics if m.metric_type == HealthMetricType.STEPS]
        if steps_data:
            total_steps = sum(m.value for m in steps_data)
            briefing.append(f"👟 Steps: {total_steps:,.0f}")
        
        # Heart health
        hr_data = [m for m in yesterday_metrics if m.metric_type == HealthMetricType.HEART_RATE]
        if hr_data:
            avg_hr = np.mean([m.value for m in hr_data])
            briefing.append(f"❤️ Avg Heart Rate: {avg_hr:.0f} bpm")
        
        # Key recommendations
        insights = self.insight_generator.generate_insights(yesterday_metrics, self.user_profile)
        if insights:
            briefing.append("\n🎯 Today's Focus:")
            for insight in insights[:2]:
                briefing.append(f"• {insight.recommendations[0]}")
        
        return "\n".join(briefing)

In [ ]:
# Demo: Create sample health data and run analysis
def generate_sample_health_data(days: int = 30) -> List[HealthMetric]:
    """Generate sample health data for demonstration"""
    metrics = []
    base_date = datetime.now() - timedelta(days=days)
    
    for day in range(days):
        current_date = base_date + timedelta(days=day)
        
        # Daily metrics
        # Steps (with weekend variation)
        is_weekend = current_date.weekday() >= 5
        base_steps = 8000 if not is_weekend else 6000
        steps = base_steps + np.random.normal(0, 2000)
        metrics.append(HealthMetric(
            metric_type=HealthMetricType.STEPS,
            value=max(0, steps),
            timestamp=current_date.replace(hour=20),
            unit="steps"
        ))
        
        # Sleep (night before)
        sleep_hours = 7.5 + np.random.normal(0, 1)
        metrics.append(HealthMetric(
            metric_type=HealthMetricType.SLEEP,
            value=max(4, min(12, sleep_hours)),
            timestamp=current_date.replace(hour=7),
            unit="hours"
        ))
        
        # Heart rate (multiple readings per day)
        for hour in [8, 12, 16, 20]:
            base_hr = 65 if hour == 8 else 75
            hr = base_hr + np.random.normal(0, 10)
            metrics.append(HealthMetric(
                metric_type=HealthMetricType.HEART_RATE,
                value=max(45, min(180, hr)),
                timestamp=current_date.replace(hour=hour),
                unit="bpm"
            ))
        
        # Weight (weekly)
        if day % 7 == 0:
            weight = 70 + (day / 30) * 0.5 + np.random.normal(0, 0.5)
            metrics.append(HealthMetric(
                metric_type=HealthMetricType.WEIGHT,
                value=weight,
                timestamp=current_date.replace(hour=8),
                unit="kg"
            ))
        
        # Blood pressure (every 3 days)
        if day % 3 == 0:
            systolic = 120 + np.random.normal(0, 10)
            metrics.append(HealthMetric(
                metric_type=HealthMetricType.BLOOD_PRESSURE,
                value=systolic,
                timestamp=current_date.replace(hour=9),
                unit="mmHg",
                metadata={"diastolic": 80 + np.random.normal(0, 5)}
            ))
        
        # Hydration
        water_ml = 2000 + np.random.normal(0, 500)
        metrics.append(HealthMetric(
            metric_type=HealthMetricType.HYDRATION,
            value=max(500, water_ml),
            timestamp=current_date.replace(hour=21),
            unit="ml"
        ))
    
    # Add some anomalies
    # Poor sleep night
    metrics.append(HealthMetric(
        metric_type=HealthMetricType.SLEEP,
        value=4.5,
        timestamp=datetime.now() - timedelta(days=5, hours=7),
        unit="hours"
    ))
    
    # High heart rate
    metrics.append(HealthMetric(
        metric_type=HealthMetricType.HEART_RATE,
        value=110,
        timestamp=datetime.now() - timedelta(days=3, hours=14),
        unit="bpm"
    ))
    
    return metrics

# Run the demo
print("🏥 Personal Health Optimization Agent Demo")
print("="*50)

# Initialize agent
agent = PersonalHealthOptimizationAgent()

# Generate sample data
print("\n📊 Generating 30 days of health data...")
sample_metrics = generate_sample_health_data(30)
print(f"Generated {len(sample_metrics)} health data points")

# Analyze health data
print("\n🔍 Analyzing health data...")
report = agent.analyze_health_data(sample_metrics)

# Display summary
print("\n📋 Health Report Summary:")
summary = report['summary']
print(f"• Overall Health Score: {summary['overall_health_score']:.1f}/100 ({summary['status']})")
print(f"• Critical Issues: {summary['critical_issues']}")
print(f"• High Priority Issues: {summary['high_priority_issues']}")
print(f"• Total Insights Generated: {summary['total_insights']}")
print(f"• Metrics Monitored: {summary['monitored_metrics']}")

# Display top insights
print("\n🎯 Top Health Insights:")
for i, insight in enumerate(report['insights'][:5], 1):
    print(f"\n{i}. {insight.title}")
    print(f"   Category: {insight.category} | Severity: {insight.severity}")
    print(f"   {insight.description}")
    print("   Recommendations:")
    for rec in insight.recommendations[:2]:
        print(f"   • {rec}")

# Display patterns
print("\n📈 Health Patterns Detected:")
if 'trends' in report['patterns']:
    for metric_type, trend in report['patterns']['trends'].items():
        print(f"• {metric_type.replace('_', ' ').title()}: {trend['direction']} trend")

# Display risk assessment
print("\n⚠️ Health Risk Assessment:")
risks = report['risk_assessment']
print(f"• Overall Risk Level: {risks['overall_risk_level'].upper()}")
for risk_type in ['cardiovascular', 'metabolic', 'sleep_disorder', 'stress']:
    if risk_type in risks and isinstance(risks[risk_type], dict):
        risk_data = risks[risk_type]
        print(f"• {risk_type.replace('_', ' ').title()} Risk: {risk_data['level']}")

# Display recommendations
print("\n💡 Top Recommendations:")
for i, rec in enumerate(report['recommendations'][:5], 1):
    print(f"{i}. {rec['recommendation']} [{rec['priority']}]")

# Generate daily briefing
print("\n" + "="*50)
print(agent.generate_daily_briefing(sample_metrics))

# Visualizations
print("\n📊 Generating visualizations...")

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Health Metrics Dashboard', fontsize=16)

# Plot 1: Daily steps trend
steps_data = [(m.timestamp, m.value) for m in sample_metrics if m.metric_type == HealthMetricType.STEPS]
if steps_data:
    dates, values = zip(*sorted(steps_data))
    axes[0, 0].plot(dates, values, 'b-', alpha=0.6)
    axes[0, 0].axhline(y=10000, color='g', linestyle='--', label='Goal')
    axes[0, 0].set_title('Daily Steps')
    axes[0, 0].set_ylabel('Steps')
    axes[0, 0].legend()
    axes[0, 0].tick_params(axis='x', rotation=45)

# Plot 2: Sleep pattern
sleep_data = [(m.timestamp, m.value) for m in sample_metrics if m.metric_type == HealthMetricType.SLEEP]
if sleep_data:
    dates, values = zip(*sorted(sleep_data))
    axes[0, 1].plot(dates, values, 'purple', alpha=0.6)
    axes[0, 1].axhline(y=8, color='g', linestyle='--', label='Recommended')
    axes[0, 1].set_title('Sleep Duration')
    axes[0, 1].set_ylabel('Hours')
    axes[0, 1].legend()
    axes[0, 1].tick_params(axis='x', rotation=45)

# Plot 3: Heart rate distribution
hr_values = [m.value for m in sample_metrics if m.metric_type == HealthMetricType.HEART_RATE]
if hr_values:
    axes[1, 0].hist(hr_values, bins=20, color='red', alpha=0.6, edgecolor='black')
    axes[1, 0].axvline(x=np.mean(hr_values), color='darkred', linestyle='--', label=f'Avg: {np.mean(hr_values):.0f}')
    axes[1, 0].set_title('Heart Rate Distribution')
    axes[1, 0].set_xlabel('BPM')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].legend()

# Plot 4: Health score components
categories = ['Steps', 'Sleep', 'Heart Rate', 'Hydration']
scores = [75, 85, 90, 60]  # Example scores
colors = ['green' if s >= 80 else 'orange' if s >= 60 else 'red' for s in scores]
axes[1, 1].bar(categories, scores, color=colors, alpha=0.7)
axes[1, 1].set_title('Health Score Breakdown')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_ylim(0, 100)
axes[1, 1].axhline(y=80, color='g', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n✅ Health analysis complete!")

## Production Deployment Considerations

### Data Integration
- Apple HealthKit integration for iOS users
- Google Fit API for Android users
- Wearable device APIs (Fitbit, Garmin, Whoop)
- Medical device integrations (blood pressure monitors, glucose meters)

### Privacy and Security
- HIPAA compliance for health data
- End-to-end encryption
- User consent management
- Data anonymization

### Advanced Features
- Machine learning models for personalized insights
- Integration with healthcare providers
- Medication tracking and reminders
- Social features for accountability
- Emergency detection and alerts

### Scaling Considerations
- Time-series database for efficient metric storage
- Real-time anomaly detection pipeline
- Personalized ML model training
- Multi-language support